# eBiblio Book Loan Automation (Google Colab)

This notebook prepares and runs `book-loan.robot` in a Google Colab runtime. Run the cells from top to bottom. You will upload the Robot suite and your private `credentials.resource` file; the notebook does not store credentials in its source.

> **Live action warning:** The live run logs in to eBiblio and may place a reservation or complete a real loan. It can wait up to four hours for today's newspaper. The live run is disabled by default; review the workflow and enable the explicit confirmation only when you intend to proceed. Keep credentials and generated reports private.

### 1. Inicializar Directorio de Trabajo Local (Sin Google Drive)
Para evitar dependencias persistentes, el cuaderno trabajará por defecto de forma temporal en el almacenamiento local rápido de Google Colab (`/content/book-loan`).

In [11]:
from pathlib import Path

# Define el directorio de trabajo local temporal en Colab
WORK_DIR = Path('/content/book-loan')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Directorio de trabajo local preparado: {WORK_DIR}")

Directorio de trabajo local preparado: /content/book-loan


## 2. Upload the Robot suite and credentials

Upload both `book-loan.robot` and your own `credentials.resource` when prompted. The credentials file must define non-empty `${USUARIO}` and `${PASSWORD}` variables. The cell writes the files into the temporary Colab runtime and does not display their contents.

In [12]:
import urllib.request
from pathlib import Path
import os
import re

# ==================== PARÁMETROS FORMULARIO COLAB ====================
#@markdown ### Configuración manual de GitHub:
GITHUB_OWNER = "mrg321" #@param {type:"string"}
GITHUB_REPO = "autobiblio" #@param {type:"string"}
GITHUB_BRANCH = "main" #@param {type:"string"}
# =====================================================================

# Forzamos que la variable WORK_DIR apunte de manera local y temporal
WORK_DIR = Path('/content/book-loan')
WORK_DIR.mkdir(parents=True, exist_ok=True)

def obtener_repo_desde_notebook():
    try:
        from google.colab import _message
        nb_json = _message.blocking_request('get_ipynb', request='', timeout_sec=5)
        if nb_json and 'metadata' in nb_json:
            colab_meta = nb_json['metadata'].get('colab', {})
            repo_history = colab_meta.get('provenance', [])
            for p in repo_history:
                if 'github.com' in p.get('url', ''):
                    return p['url']
    except Exception:
        pass
    return None

repo_detectado = obtener_repo_desde_notebook()

owner = GITHUB_OWNER
repo = GITHUB_REPO
branch = GITHUB_BRANCH

if repo_detectado:
    match = re.search(r"github\.com/([^/]+)/([^/]+)/(?:blob|raw)/([^/]+)/", repo_detectado)
    if match:
        owner, repo, branch = match.groups()
        print(f"✓ Repositorio autodetectado con éxito: {owner}/{repo} (rama: {branch})")
    else:
        print("⚠ Se detectó una URL de GitHub pero no se pudo parsear automáticamente.")
else:
    print("ℹ No se pudo autodetectar el cuaderno de forma directa. Utilizando los valores del formulario de la derecha.")

GITHUB_ROBOT_URL = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/book-loan.robot"
GITHUB_CREDENTIALS_URL = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/credentials.resource"

def descargar_desde_github(url, destino):
    try:
        print(f"Descargando desde GitHub: {url}...")
        req = urllib.request.Request(
            url,
            headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        )
        with urllib.request.urlopen(req) as response, open(destino, 'wb') as out_file:
            out_file.write(response.read())
        print(f"✓ Archivo guardado con éxito en: {destino}")
    except Exception as e:
        raise RuntimeError(
            f"No se pudo descargar el archivo de GitHub.\n"
            f"Ruta: {url}\nError: {e}"
        )

# Guardamos de forma local temporal
robot_path = WORK_DIR / "book-loan.robot"
credentials_path = WORK_DIR / "credentials.resource"

descargar_desde_github(GITHUB_ROBOT_URL, robot_path)
descargar_desde_github(GITHUB_CREDENTIALS_URL, credentials_path)

os.chmod(credentials_path, 0o600)

# Validaciones locales rápidas
robot_source = robot_path.read_text(encoding="utf-8")
if not re.search(r"(?mi)^\s*Resource\s+credentials\.resource\s*$", robot_source):
    raise ValueError("El archivo book-loan.robot descargado debe importar credentials.resource")

credentials_source = credentials_path.read_text(encoding="utf-8")
for variable in ("USUARIO", "PASSWORD"):
    if not re.search(rf"(?m)^\$\{{{variable}\}}\s+\S", credentials_source):
        raise ValueError(f"El archivo credentials.resource debe definir ${{{variable}}} con un valor válido.")

print(f"\n¡Completado! Ambos archivos se han descargado de forma local temporal en: {WORK_DIR}")

ℹ No se pudo autodetectar el cuaderno de forma directa. Utilizando los valores del formulario de la derecha.
Descargando desde GitHub: https://raw.githubusercontent.com/mrg321/autobiblio/main/book-loan.robot...
✓ Archivo guardado con éxito en: /content/book-loan/book-loan.robot
Descargando desde GitHub: https://raw.githubusercontent.com/mrg321/autobiblio/main/credentials.resource...
✓ Archivo guardado con éxito en: /content/book-loan/credentials.resource

¡Completado! Ambos archivos se han descargado de forma local temporal en: /content/book-loan


### 2.1. Editar Credenciales Localmente (Formulario rápido)

In [13]:
#@markdown Introduce aquí tus credenciales para actualizar el archivo temporal `credentials.resource`:
NUEVO_USUARIO = "9774335" #@param {type:"string"}
NUEVA_CONTRASEÑA = "6563" #@param {type:"string"}

from pathlib import Path
import os

WORK_DIR = Path('/content/book-loan')
credentials_path = WORK_DIR / "credentials.resource"

# Generamos la estructura requerida por Robot Framework
content = f"""*** Variables ***
${{USUARIO}}     {NUEVO_USUARIO}
${{PASSWORD}}    {NUEVA_CONTRASEÑA}
"""

# Sobrescribimos el archivo local
credentials_path.write_text(content, encoding="utf-8")
os.chmod(credentials_path, 0o600)

print("✓ ¡Archivo credentials.resource actualizado con éxito en el almacenamiento temporal!")


✓ ¡Archivo credentials.resource actualizado con éxito en el almacenamiento temporal!


## 3. Check and install the pinned Python dependencies

The project pins Robot Framework and Browser Library in `requirements.txt`. This cell checks the runtime first and installs only missing or mismatched versions.

In [14]:
import importlib.metadata
import sys
from IPython import get_ipython

required_versions = {
    "robotframework": "7.5",
    "robotframework-browser": "20.6.0",
}
packages_to_install = []
for package, required_version in required_versions.items():
    try:
        installed_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    print(f"{package}: {installed_version or 'not installed'} (required {required_version})")
    if installed_version != required_version:
        packages_to_install.append(f"{package}=={required_version}")

if packages_to_install:
    get_ipython().run_line_magic("pip", "install " + " ".join(packages_to_install))
else:
    print("Pinned versions are already installed.")

for package, required_version in required_versions.items():
    installed_version = importlib.metadata.version(package)
    if installed_version != required_version:
        raise RuntimeError(
            f"{package} {installed_version} is installed, but {required_version} is required."
        )
print("Python dependencies are ready.")

robotframework: 7.5 (required 7.5)
robotframework-browser: 20.6.0 (required 20.6.0)
Pinned versions are already installed.
Python dependencies are ready.


## 4. Prepare Node.js for Browser Library

Browser Library uses Node.js. Google Colab normally includes it; install the system packages only if either `node` or `npm` is missing.

In [15]:
import shutil
import subprocess
import sys

if not sys.platform.startswith("linux"):
    raise RuntimeError("Use a Linux-based Google Colab runtime for this notebook.")

if shutil.which("node") is None or shutil.which("npm") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "nodejs", "npm"], check=True)

for executable in ("node", "npm"):
    version = subprocess.run(
        [executable, "--version"], check=True, capture_output=True, text=True
    ).stdout.strip()
    print(f"{executable}: {version}")

node: v20.19.0
npm: 10.8.2


## 5. Install shared libraries

In [16]:
import subprocess
print("Instalando las librerías compartidas del sistema requeridas por Playwright/Chromium...")
try:
    # Ejecuta el comando oficial de Playwright para instalar dependencias de sistema en Ubuntu
    subprocess.run(["pip", "install", "playwright"], check=True)
    subprocess.run(["playwright", "install-deps"], check=True)
    print("\n✓ ¡Librerías del sistema instaladas correctamente!")
except Exception as e:
    print(f"\n⚠ Error instalando dependencias del sistema: {e}")
    print("Intentando instalación manual alternativa...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run([
        "apt-get", "install", "-y", "-qq",
        "libatk-1.0-0", "libatk-bridge2.0-0", "libxkbcommon0", "libxcomposite1",
        "libxdamage1", "libxrandr2", "libgbm1", "libasound2", "libpango-1.0-0",
        "libcairo2", "libatspi0"
    ], check=True)
    print("\n✓ ¡Librerías del sistema instaladas manualmente!")

Instalando las librerías compartidas del sistema requeridas por Playwright/Chromium...

✓ ¡Librerías del sistema instaladas correctamente!


## 6. Install Chromium for Browser Library

This initializes the Chromium browser that the Robot suite will use in headless mode.

In [17]:
subprocess.run(
    ["rfbrowser", "init", "chromium"],
    cwd=WORK_DIR,
    check=True,
)
print("Chromium is ready for the Robot Framework Browser Library.")

Chromium is ready for the Robot Framework Browser Library.


## 7. Validate without logging in

The dry run checks suite parsing, resource imports, and keyword resolution. It does not open a browser or submit a loan/reservation.

In [18]:
dry_run_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "robot",
        "--dryrun",
        "--output",
        "NONE",
        "--log",
        "NONE",
        "--report",
        "NONE",
        "--variable",
        "HEADLESS:True",
        str(WORK_DIR / "book-loan.robot"),
    ],
    cwd=WORK_DIR,
    check=False,
)
if dry_run_result.returncode != 0:
    raise RuntimeError(
        f"Robot Framework dry run failed with exit code {dry_run_result.returncode}. "
        "Resolve the reported issue before enabling the live run."
    )
print("Dry run passed; no live browser actions were performed.")

Dry run passed; no live browser actions were performed.


## 8. Optional live loan/reservation attempt

Only change `RUN_LIVE_BOOK_LOAN` to `True` after reviewing this Robot workflow and confirming that you want it to log in and possibly complete a real loan or reservation. Then run this cell. The suite may wait up to four hours for today's edition.

In [19]:
RUN_LIVE_BOOK_LOAN = True

if not RUN_LIVE_BOOK_LOAN:
    print(
        "Live automation was skipped. Set RUN_LIVE_BOOK_LOAN = True in this cell "
        "only when you intend to allow a real eBiblio loan or reservation attempt."
    )
else:
    print(
        "Starting live eBiblio automation with an extended timeout. This may complete a loan or place a reservation."
    )
    live_result = subprocess.run(
        [
            sys.executable,
            "-m",
            "robot",
            "--outputdir",
            str(WORK_DIR / "results"),
            "--variable",
            "HEADLESS:True",
            "--variable",
            "PLAYWRIGHT_TIMEOUT:30s",
            str(WORK_DIR / "book-loan.robot"),
        ],
        cwd=WORK_DIR,
        check=False,
    )
    if live_result.returncode != 0:
        raise RuntimeError(
            f"Robot Framework live run failed with exit code {live_result.returncode}. "
            "Inspect the generated report and log in the private runtime."
        )
    print(f"Run artifacts were written to {WORK_DIR / 'results'}.")

Starting live eBiblio automation with an extended timeout. This may complete a loan or place a reservation.
Run artifacts were written to /content/book-loan/results.


## 9. Extract last result from output.xml

In [21]:
import xml.etree.ElementTree as ET
from pathlib import Path

# Buscar el archivo output.xml en el directorio local de ejecución
WORK_DIR = Path('/content/book-loan')
output_path = WORK_DIR / "results" / "output.xml"

print(f"Analizando archivo de resultados en: {output_path}\n")

if output_path.exists():
    try:
        tree = ET.parse(output_path)
        root = tree.getroot()

        # Recopilar todos los elementos <msg> del XML de ejecución
        all_messages = []
        for msg in root.iter('msg'):
            level = msg.attrib.get('level', 'INFO')
            text = msg.text.strip() if msg.text else ""
            timestamp = msg.attrib.get('timestamp', '')
            if text:
                all_messages.append({
                    'level': level,
                    'text': text,
                    'timestamp': timestamp
                })

        if all_messages:
            # Tomar los últimos 4 mensajes del log
            last_4_messages = all_messages[-4:]
            print(f"📋 ÚLTIMOS 4 MENSAJES REGISTRADOS EN EL LOG (Total: {len(all_messages)}):\n")
            for i, msg_data in enumerate(last_4_messages, 1):
                icon = "❌" if msg_data['level'] == 'FAIL' else "ℹ️"
                if msg_data['level'] == 'PASS':
                    icon = "✅"

                print(f"  {i}. [{msg_data['level']}] {icon} {msg_data['text']}")
                print("  " + "-" * 40)
        else:
            print("No se encontraron mensajes de log detallados en el archivo XML.")

    except Exception as e:
        print(f"⚠ Error al procesar el archivo XML: {e}")
else:
    print("❌ No se encontró el archivo output.xml local. Asegúrate de que la automatización se haya completado.")

Analizando archivo de resultados en: /content/book-loan/results/output.xml

📋 ÚLTIMOS 4 MENSAJES REGISTRADOS EN EL LOG (Total: 28):

  1. [FAIL] ❌ TimeoutError: locator.waitFor: Timeout 5000ms exceeded.
Call log:
  - waiting for locator('button:has-text("Reservar")') to be visible
  ----------------------------------------
  2. [INFO] ℹ️ ${is_reservable} = False
  ----------------------------------------
  3. [INFO] ℹ️ El periódico no está disponible ni es reservable.
  ----------------------------------------
  4. [INFO] ℹ️ Closed browser
  ----------------------------------------


## 10. Optional: Network diagnosis from Colab

In [ ]:
# ==================== PASO NUEVO: RESPALDO EN GOOGLE DRIVE ====================
#@markdown ### ¿Deseas guardar los resultados permanentemente en tu Google Drive?
GUARDAR_EN_DRIVE = False #@param {type:"boolean"}

if GUARDAR_EN_DRIVE:
    try:
        from google.colab import drive
        import shutil
        from pathlib import Path

        print("Iniciando proceso de montaje de Google Drive...")
        drive.mount('/content/drive')

        DRIVE_DIR = Path('/content/drive/MyDrive/ebiblio-book-loan')
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)

        local_results = Path('/content/book-loan/results')

        if local_results.exists():
            shutil.copytree(local_results, DRIVE_DIR / 'results', dirs_exist_ok=True)
            print(f"✓ ¡Éxito! Copia de resultados guardada en Google Drive: {DRIVE_DIR / 'results'}")
        else:
            print("⚠ No se encontraron resultados locales para respaldar en la ruta /content/book-loan/results")
    except Exception as e:
        print(f"❌ Error al guardar datos en Drive: {e}")
else:
    print("Guardado en Drive omitido por el usuario. Los archivos solo residen en el almacenamiento temporal local.")

## 11. View private run reports

After a live run, these links open the Robot Framework report and log in the Colab runtime. They may contain account or session details; do not share them or save them in a shared notebook.

In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display, HTML

WORK_DIR = Path('/content/book-loan')
local_results = WORK_DIR / "results"

report_names = ("report.html", "log.html")
reports_found = False

print("=== REPORTES DE LA AUTOMATIZACIÓN (LOCALES) ===\n")
for name in report_names:
    local_file = local_results / name
    if local_file.is_file():
        print(f"Archivo: {name}")
        print("  - Enlace de descarga local:")
        display(FileLink(str(local_file)))
        print("-" * 50)
        reports_found = True

if not reports_found:
    print(f"No se encontraron reportes locales en: {local_results}.")

## 12. Cleanup Temporary Files
Run this cell to delete the sensitive credential files and execution reports from your Google Drive after you have finished.

In [ ]:
import shutil
from pathlib import Path

# Confirmación para realizar la limpieza temporal local
CONFIRM_DELETE = True

if CONFIRM_DELETE:
    # 1. Limpieza del entorno local temporal de Colab
    local_project = Path('/content/book-loan')
    if local_project.exists():
        shutil.rmtree(local_project)
        print("✓ Se ha eliminado el directorio de proyecto local temporal: /content/book-loan")

    # Eliminar otros posibles residuos sueltos en /content
    local_content_dir = Path('/content')
    for pattern in ("*book-loan*.robot", "*credentials*.resource"):
        for f in local_content_dir.glob(pattern):
            if f.is_file():
                f.unlink()

    # Limpieza de resultados temporales si existiesen fuera de la carpeta
    local_results = Path('/content/results')
    if local_results.exists():
        shutil.rmtree(local_results)

    print("Limpieza temporal local completada correctamente.")
else:
    print("Limpieza omitida.")